# Gikomba Stall Analysis: Wanjiku's Fresh-Produce Business
**Analyst:** Stanley Wanyoike Ngugi | **Cohort:** <1Mill Devs Cohort 2026>

Wanjiku runs a fresh-produce stall at Gikomba Market. This notebook uses core Python only (no pandas/NumPy) to answer: How is the business doing? Is it growing? What should be restocked, and where should she focus?

In [8]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics
I use `sum()`, `max()` and `min()` for the totals. `list.index()` gives the position of the best and worst days, and the day name comes from `days[position]`. The week number is `position // 7 + 1`.

In [9]:
total_sales = sum(daily_sales)
avg_sales = total_sales / len(daily_sales)
best_sales = max(daily_sales)
worst_sales = min(daily_sales)

best_idx = daily_sales.index(best_sales)
worst_idx = daily_sales.index(worst_sales)
best_day, best_week = days[best_idx], best_idx // 7 + 1
worst_day, worst_week = days[worst_idx], worst_idx // 7 + 1

print(f"Total: KES {total_sales:,} | Average: KES {avg_sales:,.2f}")
print(f"Best: KES {best_sales:,} ({best_day}, week {best_week})")
print(f"Worst: KES {worst_sales:,} ({worst_day}, week {worst_week})")

Total: KES 46,450 | Average: KES 3,317.86
Best: KES 5,600 (Sat, week 2)
Worst: KES 1,950 (Tue, week 1)


## Task 2: Week-on-week growth
`week_totals()` slices the list into positions 0–6 and 7–13 and returns both totals as a tuple. Growth % = (week 2 − week 1) ÷ week 1 × 100.

In [10]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a list of 14 daily sales."""
    return sum(sales[:7]), sum(sales[7:])

week1, week2 = week_totals(daily_sales)
growth_pct = (week2 - week1) / week1 * 100
print(f"Week 1: KES {week1:,} | Week 2: KES {week2:,} | Growth: {growth_pct:.1f}%")

Week 1: KES 22,050 | Week 2: KES 24,400 | Growth: 10.7%


## Task 3: Classifying days
`classify_day()` compares each day to the 14-day average. Above 120% of the average is "Strong", below 80% is "Weak", anything else is "Normal". A dictionary counts how many days fall in each category.

In [11]:
def classify_day(amount, average):
    """Return 'Strong' if amount is >20% above average, 'Weak' if >20% below, else 'Normal'."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"

day_counts = {}
for day, amount in zip(days, daily_sales):
    label = classify_day(amount, avg_sales)
    day_counts[label] = day_counts.get(label, 0) + 1

print(day_counts)

{'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays vs weekends
Position `i % 7` is 0–4 for Mon–Fri and 5–6 for Sat–Sun, so I split the sales on that and compare the averages.

In [12]:
weekday_sales = [s for i, s in enumerate(daily_sales) if i % 7 < 5]
weekend_sales = [s for i, s in enumerate(daily_sales) if i % 7 >= 5]

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday avg: KES {weekday_avg:,.2f} | Weekend avg: KES {weekend_avg:,.2f}")
print(f"Weekends sell {weekend_uplift:.1f}% more than weekdays")

Weekday avg: KES 2,685.00 | Weekend avg: KES 4,900.00
Weekends sell 82.5% more than weekdays


## Task 5: Stock reorder list and gross margins
`products_to_reorder()` loops over `stock.items()` and keeps products where `qty <= reorder`. Gross margin % = (price − cost) ÷ price × 100 for each product.

In [13]:
def products_to_reorder(stock_dict):
    """Return a list of product names whose quantity is at or below the reorder level."""
    return [name for name, info in stock_dict.items() if info["qty"] <= info["reorder"]]

def gross_margins(stock_dict):
    """Return a dict of product name -> gross margin %, calculated as (price - cost) / price * 100."""
    return {name: (info["price"] - info["cost"]) / info["price"] * 100
            for name, info in stock_dict.items()}

reorder_list = products_to_reorder(stock)
margins = gross_margins(stock)
print("Reorder now:", reorder_list)
for name, m in margins.items():
    print(f"{name:<10} {m:.1f}%")

Reorder now: ['Tomatoes', 'Sukuma']
Tomatoes   33.3%
Onions     30.0%
Sukuma     40.0%
Potatoes   33.3%
Avocado    42.9%


## Task 6: Stall Performance Report
All figures from Tasks 1–5 are printed with f-strings, with KES amounts using thousands separators and aligned columns.

In [14]:
line = "=" * 46
print(line)
print(f"{'STALL PERFORMANCE REPORT':^46}")
print(line)
print(f"{'Total sales (14 days):':<28}KES {total_sales:>10,}")
print(f"{'Average daily sales:':<28}KES {avg_sales:>10,.2f}")
print(f"{'Best day:':<28}{best_day} wk{best_week}, KES {best_sales:,}")
print(f"{'Worst day:':<28}{worst_day} wk{worst_week}, KES {worst_sales:,}")
print("-" * 46)
print(f"{'Week 1 total:':<28}KES {week1:>10,}")
print(f"{'Week 2 total:':<28}KES {week2:>10,}")
print(f"{'Week-on-week growth:':<28}{growth_pct:>14.1f}%")
print("-" * 46)
print(f"{'Weekday average:':<28}KES {weekday_avg:>10,.2f}")
print(f"{'Weekend average:':<28}KES {weekend_avg:>10,.2f}")
print(f"{'Weekend uplift:':<28}{weekend_uplift:>14.1f}%")
print("-" * 46)
print("Day classification:")
for label in ["Strong", "Normal", "Weak"]:
    print(f"  {label:<10}{day_counts.get(label, 0):>3} days")
print("-" * 46)
print(f"{'Product':<12}{'Stock':>8}{'Reorder':>9}{'Margin':>9}")
for name, info in stock.items():
    flag = "  << REORDER" if name in reorder_list else ""
    print(f"{name:<12}{info['qty']:>6}kg{info['reorder']:>7}kg{margins[name]:>8.1f}%{flag}")
print(line)

           STALL PERFORMANCE REPORT           
Total sales (14 days):      KES     46,450
Average daily sales:        KES   3,317.86
Best day:                   Sat wk2, KES 5,600
Worst day:                  Tue wk1, KES 1,950
----------------------------------------------
Week 1 total:               KES     22,050
Week 2 total:               KES     24,400
Week-on-week growth:                  10.7%
----------------------------------------------
Weekday average:            KES   2,685.00
Weekend average:            KES   4,900.00
Weekend uplift:                       82.5%
----------------------------------------------
Day classification:
  Strong      4 days
  Normal      4 days
  Weak        6 days
----------------------------------------------
Product        Stock  Reorder   Margin
Tomatoes        18kg     25kg    33.3%  << REORDER
Onions          40kg     20kg    30.0%
Sukuma           9kg     15kg    40.0%  << REORDER
Potatoes        55kg     30kg    33.3%
Avocado         12kg   

## Task 7: Business Recommendations for Wanjiku
1. **Restock Tomatoes and Sukuma immediately.** Tomatoes are at 18 kg against a 25 kg reorder level and Sukuma at 9 kg against 15 kg, so she risks stock-outs before the weekend.
2. **Stock up before the weekend.** Weekends average KES 4,900 versus KES 2,685 on weekdays (+82.5%), so Friday should be her main buying day.
3. **Work on the weak weekdays.** Six of 14 days (all Mon, Tue and Thu) fell below KES 2,654, so a small promotion on those days could lift the average.
4. **Push Avocado and Sukuma.** They have the highest margins (42.9% and 40.0%), so featuring them earns more per kg than Onions at 30.0%.
5. **Keep the momentum.** Sales grew 10.7% week-on-week (KES 22,050 to KES 24,400), so she should keep tracking sales daily to confirm the trend.